In [ ]:
##--Python’s built-in string module, which provides : Utilities and constants for working with text.
#example: string.punctuation → list of punctuation characters, we will use this for text preprocessing, such as removing punctuation.
#--Python’s random number generator module which provides - Functions for generating random values.
 #example: 
 #random.choice(list) → choose random element
 #random.randint(a,b) → random integer
 #random.random() → random float
 #random.shuffle(list) → shuffle elements, 
 #Commonly used for: random sampling, generating random text, language models, probabilistic selection
#--NLTK: A Python library used for Natural Language Processing (NLP) tasks.
#It supports: Tokenization, POS tagging, Language models, Corpora, Text classification, Parsing
#--FreqDist: Counts how often words appear in a text, Useful for: word frequency analysis, language modeling, text statistics
#--Brown corpus: A large collection of English text compiled by Brown University.
#It contains: ~1 million words: multiple categories like - news, fiction, romance, government, religion
#Used to Used to: train language models, calculate word probabilities, study English usage
#--Counter: Counts occurrences of elements (word counting).defaultdict : A dictionary that automatically creates default values (for grouping data , building language models)
#--ngrams: An n-gram is a sequence of n consecutive words.
'''
Unigram (1-gram)
I
love
machine
learning

Bigram (2-gram)
(I, love)
(love, machine)
(machine, learning)

Trigram (3-gram)
(I, love, machine)
(love, machine, learning)
'''


'\nUnigram (1-gram)\nI\nlove\nmachine\nlearning\n\nBigram (2-gram)\n(I, love)\n(love, machine)\n(machine, learning)\n\nTrigram (3-gram)\n(I, love, machine)\n(love, machine, learning)\n'

In [2]:
#!pip install nltk
import string
import random
import nltk
from nltk import FreqDist
from nltk.corpus import brown
from collections import defaultdict, Counter
from nltk.util import ngrams

In [3]:

nltk.download('punkt') #downloads the Punkt tokenizer model.
#punkt is used for sentence and word tokenization (splitting text into sentences or words), NLTK needs the pre-trained tokenizer model.
nltk.download('stopwords')
nltk.download('brown')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ajay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ajay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\Ajay\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [6]:
# Define stopwords and punctuation
#loads English stopwords from NLTK and converts them into a set, because sets are much faster for lookup.
stop_words = set(nltk.corpus.stopwords.words('english'))

#add extra punctuation characters to the existing punctuation list.
string.punctuation += '"\'-—'

#create a single list of tokens to remove from the text.
removal_list = list(stop_words) + list(string.punctuation) + ['lt', 'rt']


In [11]:
# Load sentences from the Brown corpus
sents = brown.sents()

# Initialize lists for storing n-grams
unigram = []
bigram = []
trigram = []



In [12]:
#load text from the Brown corpus and create unigram, bigram, and trigram datasets. 
#These are commonly used in n-gram language models for tasks like next-word prediction or text generation.
# Generate n-grams
for sentence in sents:
    sentence = [word.lower() for word in sentence if word not in string.punctuation]
    unigram.extend(sentence)
    #Creates pairs of consecutive words.#Padding: adds sentence boundary markers, to help model learn sentence start and end
    bigram.extend(list(ngrams(sentence, 2, pad_left=True, pad_right=True))) 
    trigram.extend(list(ngrams(sentence, 3, pad_left=True, pad_right=True)))

In [8]:
'''We are preparing data to build:
unigram language model
bigram language model
trigram language model

These are used for:
Applications
next-word prediction
autocomplete
chatbot text generation
speech recognition
spell correction'''

'We are preparing data to build:\nunigram language model\nbigram language model\ntrigram language model\n\nThese are used for:\nApplications\nnext-word prediction\nautocomplete\nchatbot text generation\nspeech recognition\nspell correction'

In [16]:
# Function to remove stopwords from n-grams
#filter out unwanted words (stopwords, punctuation, etc.) from the bigram and trigram lists that were created earlier.
#--Remove all bigrams containing stopwords
#--Remove stopwords from trigrams
def remove_stopwords(ngrams, n):
    if n == 2:
        return [(a, b) for (a, b) in ngrams if a not in removal_list and b not in removal_list]
    elif n == 3:
        return [(a, b, c) for (a, b, c) in ngrams if a not in removal_list and b not in removal_list and c not in removal_list]

# Remove stopwords from n-grams
bigram = remove_stopwords(bigram, 2)
trigram = remove_stopwords(trigram, 3)


In [21]:
# Calculate frequency distributions- how often each bigram and trigram occurs in the dataset.
#FreqDist creates a dictionary-like structure that stores the frequency of each item.
#These counts are used to compute language model probabilities.
freq_bi = FreqDist(bigram)
freq_tri = FreqDist(trigram)

In [23]:
# Create a dictionary of trigram frequencies
# Builds a data structure used for predicting the next word in a trigram language model.
#It organizes trigram frequencies so that given two words, you can easily see what words most commonly come next.
d = defaultdict(Counter)
for ngram in freq_tri:
    if None not in ngram:
        d[ngram[:-1]][ngram[-1]] += freq_tri[ngram]

In [31]:
d[('world','war')]

Counter({'2': 35, '1': 13, '3': 1, "''": 1, 'st.': 1, 'baker': 1})

In [25]:
# Function to generate text
# This function generates text using the trigram language model we built, It predicts the next word based on the previous two words.
def generate_text(prefix, n=20):
    for _ in range(n):
        suffix_candidates = list(d.get(prefix, Counter()).elements())
        if not suffix_candidates:
            new_prefix = random.choice(unigram), random.choice(unigram)
            yield new_prefix[0]  # Yield the first word of the new prefix
            prefix = new_prefix
        else:
            suffix = random.choice(suffix_candidates)
            yield suffix
            prefix = (*prefix[1:], suffix)

In [26]:
# Generate text - runs the text generator and prints the generated sentence.
prefix = ("he", "said")
generated_text = list(generate_text(prefix))
if generated_text:
    print(" ".join(generated_text))
else:
    print("No text generated.")

america the to of to other they '' road the able he is that clergyman other the pittsburgh '' the


In [14]:
#Optimizing
#1.Add probability calculation instead of frequency sampling
#Now model chooses  a word using- random.choice(suffix_candidates)
#This works because repeated words approximate probability, but a cleaner approach is to calculate probabilities explicitly.

#2.Model fails when it sees unseen word combinations.
#example: prefix = ("computer", "banana")
#No trigram exists → model resets randomly.
#So we can Use smoothing techniques:
'''
Common ones:
Laplace Smoothing
Add-k smoothing
Good-Turing smoothing
Kneser-Ney smoothing
'''

#3.Use sentence start/end tokens,Instead of padding with None, use explicit tokens

#4.Generate full sentences instead of fixed length

#5.Improve prefix selection, instead of manually set : prefix = ("he","said")

#6.Keep stopwords for language models
#Removing stopwords is useful for topic analysis, but not ideal for language generation.

#7.Add temperature/randomness control
#In modern text generation (like GPT), we control randomness using temperature.
#we can implement something similar with weighted sampling.

#8. Evaluate the model using perplexity.It is a common NLP metric, that tells how well the model predicts text.
#Lower perplexity → better model.

#9.Train on different corpora
'''
Instead of only Brown corpus, try:

movie scripts
news articles
books
tweets
'''

#10.Build a next-word prediction tool
#You could turn the model into a predictive keyboard.


'\nInstead of only Brown corpus, try:\n\nmovie scripts\nnews articles\nbooks\ntweets\n'

In [32]:
#Using sentence start and end tokens i.e modify sentence preprocessing
sents = brown.sents()

processed_sents = []

for sentence in sents:
    sentence = [word.lower() for word in sentence if word not in string.punctuation]
    
    # Add start and end tokens
    sentence = ['<s>', '<s>'] + sentence + ['</s>']
    
    processed_sents.append(sentence)

In [33]:
#generate n-grams from processed_sents instead of sents
#This lets the model learn sentence beginnings and endings
unigram = []
bigram = []
trigram = []

for sentence in processed_sents:
    unigram.extend(sentence)
    bigram.extend(list(ngrams(sentence, 2)))
    trigram.extend(list(ngrams(sentence, 3)))

In [34]:
#Test
print(trigram[:10])

[('<s>', '<s>', 'the'), ('<s>', 'the', 'fulton'), ('the', 'fulton', 'county'), ('fulton', 'county', 'grand'), ('county', 'grand', 'jury'), ('grand', 'jury', 'said'), ('jury', 'said', 'friday'), ('said', 'friday', 'an'), ('friday', 'an', 'investigation'), ('an', 'investigation', 'of')]


In [35]:
#Calculate True Trigram Probabilities
#Instead of relying on repeated elements, compute probabilities explicitly.
trigram_prob = {}

for (w1, w2), counter in d.items():
    total = sum(counter.values())
    trigram_prob[(w1, w2)] = {}
    
    for w3 in counter:
        trigram_prob[(w1, w2)][w3] = counter[w3] / total

In [36]:
#Test
print(trigram_prob.get(("he","said")))

{}


In [37]:
#check if prefix exists 
print(("he","said") in trigram_prob)

True


In [38]:
#find valid prefixes in the model
print(list(trigram_prob.keys())[:10])

[('world', 'war'), ('new', 'york'), ('``', 'yes'), ("drug's", 'chemical'), ('``', 'well'), ('basic', 'wage'), ('john', 'a.'), ('a.', 'notte'), ('two', 'years'), ('notte', 'jr.')]


In [39]:
#now test using one of them
print(trigram_prob.get(('world','war')))

{'2': 0.6730769230769231, '1': 0.25, '3': 0.019230769230769232, "''": 0.019230769230769232, 'st.': 0.019230769230769232, 'baker': 0.019230769230769232}


In [40]:
#Optional: For language generation, do NOT remove stopwords from trigrams.
# bigram = remove_stopwords(bigram, 2)
# trigram = remove_stopwords(trigram, 3)
#Test (None if not commented out as above)
print(trigram_prob.get(("he","said")))

{}
